# Fig. 3a: classification flowchart

The tests of `scar_metrics.classify`, in the order they are applied. Saved
to `figures/fig-03/a.*`; the paper's Fig. 3 stacks it above the panels of
`fig-03.ipynb` (`figures/fig-03/b-v.*`).

In [ ]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

from qising_networks import FIGURES_DIR as FIGURES
from qising_networks import PROJECT_ROOT
from qising_networks.plots.figure import (
    LABEL_BOLD,
    LABEL_FONT,
    MAX_FIG_WIDTH,
    journal_labels,
    save_unlabeled,
)

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

FIGURES.mkdir(exist_ok=True)
FIG_ID = "fig-03a"

# Dark2 in the flowchart's left-to-right order, indexed by class.
_dark2 = mpl.colormaps["Dark2"].colors
CLASS_COLORS = (_dark2[0], _dark2[1], _dark2[3], _dark2[4], _dark2[2])


def text_colour(rgb):
    """Black or white, whichever reads better on `rgb` (Rec. 601 luma)."""
    r, g, b = rgb
    return "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.55 else "white"


CLASS_TEXT = tuple(text_colour(c) for c in CLASS_COLORS)

In [ ]:
def add_box(
    ax,
    xy,
    w,
    h,
    heading,
    body,
    heading_dy,
    body_dy,
    facecolor="white",
    edgecolor="black",
    textcolor="black",
    linewidth=None,
):
    """Rounded box with a bold heading and a body line below it."""
    x, y = xy
    if linewidth is None:
        linewidth = mpl.rcParams["axes.linewidth"]
    box = mpatches.FancyBboxPatch(
        (x - w / 2, y - h / 2),
        w,
        h,
        boxstyle="round,pad=0,rounding_size=0.14",
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=2,
    )
    ax.add_patch(box)
    ax.text(
        x,
        y + heading_dy,
        heading,
        ha="center",
        va="center",
        color=textcolor,
        fontweight="bold",
        zorder=3,
    )
    ax.text(
        x,
        y + body_dy,
        body,
        ha="center",
        va="center",
        color=textcolor,
        fontweight="normal",
        zorder=3,
    )
    return box


def add_diamond(ax, xy, hw, hh, text, facecolor="0.93", edgecolor="black"):
    x, y = xy
    pts = [(x - hw, y), (x, y + hh), (x + hw, y), (x, y - hh)]
    diamond = mpatches.Polygon(
        pts,
        closed=True,
        linewidth=mpl.rcParams["axes.linewidth"],
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=2,
    )
    ax.add_patch(diamond)
    ax.text(x, y, text, ha="center", va="center", zorder=3)
    return diamond


def add_arrow(
    ax, start, end, label=None, label_xy=None, ha="center", va="bottom"
):
    arrow = mpatches.FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=7,
        linewidth=1.0,
        color="black",
        zorder=1,
        shrinkA=0,
        shrinkB=0,
    )
    ax.add_patch(arrow)
    if label is not None:
        if label_xy is None:
            label_xy = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)
        ax.text(
            label_xy[0],
            label_xy[1],
            label,
            ha=ha,
            va=va,
            color="0.2",
            zorder=3,
        )

In [ ]:
# Larger font (7 pt) for this schematic only.
with plt.rc_context({"font.size": 7}):
    fig, ax = plt.subplots(figsize=(MAX_FIG_WIDTH, 2.0))
    ax.set_aspect("equal")
    ax.axis("off")

    Y = 0.0  # trunk row
    hw, hh = 1.78, 1.40  # diamond half-width/half-height
    L = 0.58  # length of every arrow
    # Box content sizes in inches, converted at S inches per data unit.
    S = 0.3
    ONE_LINE = 0.0945 / S  # margin above and below each box's content
    GAP = 0.04 / S  # blank gap between a box's heading and its body
    START_HEAD_H, START_BODY_H = 0.0939 / S, 0.1200 / S
    LEAF_HEAD_H, LEAF_BODY_H = (
        0.0894 / S,
        0.2858 / S,
    )  # worst case: 3-line bodies

    start_w, start_h = 3.10, START_HEAD_H + GAP + START_BODY_H + 2 * ONE_LINE
    leaf_w, leaf_h = 3.38, LEAF_HEAD_H + GAP + LEAF_BODY_H + 2 * ONE_LINE
    start_head_dy, start_body_dy = (
        (GAP + START_BODY_H) / 2,
        -(START_HEAD_H + GAP) / 2,
    )
    leaf_head_dy, leaf_body_dy = (
        (GAP + LEAF_BODY_H) / 2,
        -(LEAF_HEAD_H + GAP) / 2,
    )

    x_start = 0.4
    x_d1 = x_start + start_w / 2 + L + hw
    x_d2 = x_d1 + 2 * hw + L
    x_d3 = x_d2 + 2 * hw + L
    x_d4 = x_d3 + 2 * hw + L
    x_plateau = x_d4 + hw + L + leaf_w / 2
    Y_LEAF = (
        Y - hh - L - leaf_h / 2
    )  # leaf row (localized / recurrence / thermalized / scar)

    ax.set_xlim(x_start - start_w / 2 - 0.13, x_plateau + leaf_w / 2 + 0.13)
    ax.set_ylim(Y_LEAF - leaf_h / 2 - 0.15, Y + hh + 0.02)

    add_box(
        ax,
        (x_start, Y),
        start_w,
        start_h,
        "Fidelity",
        r"$F(t) = \mid\!\langle \psi(0) \mid \psi(t) \rangle\!\mid^2$",
        start_head_dy,
        start_body_dy,
    )

    # The tests of scar_metrics.classify: Yes drops to a class, No continues.
    add_diamond(ax, (x_d1, Y), hw, hh, r"$1-\min_t F(t)<0.1$")
    add_diamond(ax, (x_d2, Y), hw, hh, r"$\mathrm{dim(subspace)} < 180$")
    add_diamond(
        ax,
        (x_d3, Y),
        hw,
        hh,
        r"$\frac{F_\mathrm{rev}(\boldsymbol{s}_0)}{\mathrm{med}_{\boldsymbol{s}_0}(F_\mathrm{rev}(\boldsymbol{s}_0))} < 3$",
    )
    add_diamond(
        ax,
        (x_d4, Y),
        hw,
        hh,
        r"$\frac{\min_t F(t; \boldsymbol{s}_0)}{F_\mathrm{rev}(\boldsymbol{s}_0)} < 0.1$",
    )

    # trunk arrows: No
    add_arrow(ax, (x_start + start_w / 2, Y), (x_d1 - hw, Y))
    add_arrow(
        ax,
        (x_d1 + hw, Y),
        (x_d2 - hw, Y),
        label="No",
        label_xy=((x_d1 + hw + x_d2 - hw) / 2, Y - 0.1),
        va="top",
    )
    add_arrow(
        ax,
        (x_d2 + hw, Y),
        (x_d3 - hw, Y),
        label="No",
        label_xy=((x_d2 + hw + x_d3 - hw) / 2, Y - 0.1),
        va="top",
    )
    add_arrow(
        ax,
        (x_d3 + hw, Y),
        (x_d4 - hw, Y),
        label="No",
        label_xy=((x_d3 + hw + x_d4 - hw) / 2, Y - 0.1),
        va="top",
    )

    # leaves: Yes
    leaves = [
        (x_d1, "Localized", "Never leaves home", 0),
        (
            x_d2,
            "Recurrence",
            "Subspace too small\nfor non-return to\nmean anything",
            1,
        ),
        (
            x_d3,
            "Thermalized",
            "Returns, but no\nmore than a random\nstate would",
            4,
        ),
        (x_d4, "Scar", "Anomalous revival,\nrelaxes fully\nbetween peaks", 2),
    ]
    for x, name, desc, ci in leaves:
        add_box(
            ax,
            (x, Y_LEAF),
            leaf_w,
            leaf_h,
            name,
            desc,
            leaf_head_dy,
            leaf_body_dy,
            facecolor=CLASS_COLORS[ci],
            textcolor=CLASS_TEXT[ci],
        )
        add_arrow(
            ax,
            (x, Y - hh),
            (x, Y_LEAF + leaf_h / 2),
            label="Yes",
            label_xy=(x + 0.12, (Y - hh + Y_LEAF + leaf_h / 2) / 2),
            ha="left",
        )

    # plateau scar is the last diamond's "No": stays on the trunk, to its right
    add_box(
        ax,
        (x_plateau, Y),
        leaf_w,
        leaf_h,
        "Plateau scar",
        "Anomalous revival,\nstays elevated\nbetween peaks",
        leaf_head_dy,
        leaf_body_dy,
        facecolor=CLASS_COLORS[3],
        textcolor=CLASS_TEXT[3],
    )
    add_arrow(
        ax,
        (x_d4 + hw, Y),
        (x_plateau - leaf_w / 2, Y),
        label="No",
        label_xy=((x_d4 + hw + x_plateau - leaf_w / 2) / 2, Y - 0.1),
        va="top",
    )

    # Panel letter 'a', baseline on the diamond tops.
    ax.text(
        0.0,
        Y + hh,
        journal_labels("a")[0],
        transform=mpl.transforms.blended_transform_factory(
            ax.transAxes, ax.transData
        ),
        fontfamily="Helvetica",
        fontweight="bold" if LABEL_BOLD else "normal",
        fontsize=LABEL_FONT,
        va="baseline",
        ha="left",
    )

    save_unlabeled(fig, FIGURES / "fig-03", "a", bbox_inches="tight")